# Generate Raw Sweetpotato Data

## Setup

In [1]:
# Simulates raw_genotypes.csv from raw_phenotypes.csv into data/generated/<RUN_NAME>/.
# data/raw/ is not changed unless PROMOTE_GENOTYPES_TO_RAW = True.
from datetime import datetime
from pathlib import Path
import filecmp
import importlib
import shutil
import sys

import pandas as pd
from IPython.display import display

ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "data_generation").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate src/data_generation/ from the current directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import capstone_pipeline
from src.etl_pipeline import extract_phenotype, run_etl, transform_phenotype

# --- Settings ---------------------------------------------------------------
PHENOTYPE_PATH = ROOT / "data" / "raw" / "raw_phenotypes.csv"   # the only input
RUN_NAME = "latest"
PROMOTE_GENOTYPES_TO_RAW = False   # copy the new genotypes into data/raw/ (backs up first)
# -----------------------------------------------------------------------------

RUN_DIR = ROOT / "data" / "generated" / RUN_NAME
PROJECT_GENOTYPES = ROOT / "data" / "raw" / "raw_genotypes.csv"
GENOTYPE_OUTPUT = RUN_DIR / "raw_genotypes.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not PHENOTYPE_PATH.is_file():
    raise FileNotFoundError(f"Phenotype input not found: {PHENOTYPE_PATH}")

print(f"Phenotype input: {PHENOTYPE_PATH}")
print(f"Run directory:   {RUN_DIR}")

Phenotype input: C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\data\raw\raw_phenotypes.csv
Run directory:   C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\data\generated\latest


## 1. Score phenotypes

In [2]:
# Cleans and scores the raw phenotypes in memory (same ETL the simulator uses).
scored, phenotype_report = transform_phenotype(extract_phenotype(PHENOTYPE_PATH))
print(f"Labeled clones: {len(scored)}")
display(scored[capstone_pipeline.TARGET_COLUMNS].describe().T.round(3))

Labeled clones: 54


,count,mean,std,min,25%,50%,75%,max
market_index,54.0,0.543,0.087,0.321,0.489,0.528,0.621,0.797
dri,54.0,0.497,0.158,0.194,0.368,0.475,0.637,0.880
eqi,54.0,0.646,0.091,0.358,0.598,0.645,0.703,0.823
parental_value,54.0,0.523,0.096,0.314,0.457,0.506,0.615,0.800
ayi,54.0,0.487,0.142,0.221,0.365,0.512,0.598,0.773


## 2. Simulate genotypes

In [3]:
# Embeds signal in labeled clones only; candidates get none. Fixed seed.
simulator = importlib.reload(importlib.import_module("src.data_generation.simulate_realistic_genotypes"))
simulator.PHENOTYPE_PATH = PHENOTYPE_PATH
simulator.GENOTYPE_PATH = GENOTYPE_OUTPUT
GENOTYPE_OUTPUT.unlink(missing_ok=True)  # avoid a backup copy inside the run folder
simulator.main()

Simulated genotype rows: 10005 (54 labeled clones)
Causal markers embedded per target: 25 for ['market_index', 'dri', 'eqi', 'parental_value', 'ayi']
Wrote C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\data\generated\latest\raw_genotypes.csv


## 3. Compare with data/raw/

In [4]:
# Compare generated genotype output with project genotypes and labeled clones
def normalized_ids(path):
    ids = pd.read_csv(path, usecols=["clone_id"], dtype=str)["clone_id"]
    return set(ids.str.strip().str.replace(r"\s+", " ", regex=True).str.upper())
# Normalize clone IDs for comparison
new_ids, project_ids = normalized_ids(GENOTYPE_OUTPUT), normalized_ids(PROJECT_GENOTYPES)
labeled_ids = set(scored["clone_id"])
display(pd.DataFrame([{
    "byte-identical": filecmp.cmp(GENOTYPE_OUTPUT, PROJECT_GENOTYPES, shallow=False),
    "new rows": sum(1 for _ in GENOTYPE_OUTPUT.open(encoding="utf-8")) - 1,
    "project rows": sum(1 for _ in PROJECT_GENOTYPES.open(encoding="utf-8")) - 1,
    "labeled clones in new file": len(labeled_ids & new_ids),
    "shared clone IDs": len(new_ids & project_ids),
}]))

,byte-identical,new rows,project rows,labeled clones in new file,shared clone IDs
0,True,10005,10005,54,10000


## 4. ETL and signal check

In [5]:
# Cleans the new files into the run folder, then cross-validates every model on them.
etl_check = run_etl(
    raw_phenotype_path=PHENOTYPE_PATH,
    raw_genotype_path=GENOTYPE_OUTPUT,
    clean_phenotype_path=RUN_DIR / "processed" / "phenotype_clean.csv",
    clean_genotype_path=RUN_DIR / "processed" / "genotype_clean.csv",
)
report = etl_check["report"]
print(f"Clean genotype rows: {report['genotype']['final_row_count']}, "
      f"markers: {report['genotype']['marker_count']}, "
      f"labeled clones joined: {report['overlapping_clone_ids_after_join']}")
# Display comparison of new and project genotype files
signal_check = capstone_pipeline.compare_model_families_across_targets(
    RUN_DIR / "processed" / "phenotype_clean.csv",
    RUN_DIR / "processed" / "genotype_clean.csv",
    output_dir=RUN_DIR,
)
print("Cross-validated R² on the new genotypes")
display(signal_check.pivot(index="target", columns="model_family", values="cv_r2").round(3))

Clean genotype rows: 10000, markers: 500, labeled clones joined: 54


Cross-validated R² on the new genotypes


model_family,elastic_net,mlp_baseline,ridge
target,,,
ayi,0.934,0.926,0.804
dri,0.838,0.843,0.812
eqi,0.822,0.823,0.573
market_index,0.832,0.790,0.885
parental_value,0.886,0.866,0.905


## 5. Promote genotypes (optional)

In [6]:
# Set PROMOTE_GENOTYPES_TO_RAW = True to use the new genotypes; then re-run notebooks 02 and 03.
if not PROMOTE_GENOTYPES_TO_RAW:
    print("Promotion skipped (PROMOTE_GENOTYPES_TO_RAW = False); data/raw/ was not changed.")
elif filecmp.cmp(GENOTYPE_OUTPUT, PROJECT_GENOTYPES, shallow=False):
    print("New genotypes already match data/raw/raw_genotypes.csv; nothing to promote.")
else:
    backup_dir = ROOT / "data" / "raw" / f"backup_{datetime.now():%Y%m%d_%H%M%S}"
    backup_dir.mkdir(parents=True)
    shutil.copy2(PROJECT_GENOTYPES, backup_dir / PROJECT_GENOTYPES.name)
    shutil.copy2(GENOTYPE_OUTPUT, PROJECT_GENOTYPES)
    print(f"Backed up the previous file to {backup_dir}")
    print(f"Replaced {PROJECT_GENOTYPES}; re-run notebooks 02 and 03.")

Promotion skipped (PROMOTE_GENOTYPES_TO_RAW = False); data/raw/ was not changed.
